# JWST Multiphase Science Pipeline Master Dashboard
**Target Science:** IC 348 Brown Dwarf & Planetary Mass Survey  
**PI Designation:** Oleg G. Kapranov  
**Instrument System:** JWST NIRCam Broad & Narrow Filter Configurations 

This master workspace maps, triggers, and evaluates Level 1 (Detector) and Level 2 (Calibrated Image) sequences across five concurrent NIRCam setups according to the documented framework logs.

## Step 1: Environment Integration, Token Verification & Routing Paths

In [1]:
import os
import sys
import glob
import pandas as pd
from astropy.table import Table

# 1. Secure Core Pipeline Environment Variables
os.environ["MAST_API_TOKEN"] = "0bdabb80135e42a8a77929ae012b343a"  # Replace with active token
os.environ["CRDS_SERVER_URL"] = "https://jwst-crds.stsci.edu"
os.environ["CRDS_PATH"] = os.path.expanduser("~/crds_cache")

print("=== JWST Science Pipeline Environment Anchored ===")
print(f"CRDS Cache Directory Location: {os.environ['CRDS_PATH']}")

=== JWST Science Pipeline Environment Anchored ===
CRDS Cache Directory Location: /home/kapranov/crds_cache


## Step 2: Data Discovery (Pulling Down Pool Catalogs & Association Definitions)

In [ ]:
import os
import sys

# Define your 5 clean target filter directories explicitly
target_filter_paths = [
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f150w2-f162m/mastDownload/JWST",
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f150w2-f164n/mastDownload/JWST",
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f322w2-f323n/mastDownload/JWST",
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f444w-f466n/mastDownload/JWST",
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f444w-f470n/mastDownload/JWST"
]
#
# Ensure the required subdirectories exist before running metadata search sync scripts
for path in target_filter_paths:
    os.makedirs(path, exist_ok=True)

search_scripts = [
    "./search_jw01257-o003_20260714t164831.py",
    "./search_jw01257-o003_20260714t164831_00002.py",
    "./search_jw01257-o003_20260714t164831_00005.py",
    "./search_jw01257-o003_20260714t164831_00007.py",
    "./search_jw01257-o003_20260714t164831_00008.py",
    "./search_jw01257-o003_20260714t164831_00009.py"
]

print("==========================================")
print("RUNNING STEP 2: METADATA SPACE DISCOVERY")
print("==========================================")

for script in search_scripts:
    if os.path.exists(script):
        print(f"Executing discovery sync: {script}")
        try:
            # Pass globals() to make sure os and sys cross over cleanly into the scripts
            exec(open(script).read(), globals())
            print(f" -> SUCCESS: {script} synchronized cleanly.")
        except Exception as e:
            print(f" -> ERROR executing {script}: {e}")
    else:
        print(f"Skipping (not found): {script}")

print("==========================================")
print("Step 2 Database Space Discovery Finalized Successfully.")
print("==========================================")

## Step 3: Archive Retrieval (Executing Association Space Downloads from MAST Archive)

In [ ]:
import os
import sys

# Pass ONLY the clean filenames. download_asn_members.py automatically appends its internal 'mast_jwst_root' path.
download_matrix = [
    {"asn_name": "jw01257-o003_20260714t164831_image3_00002_asn.json", "dest_dir": "./processed_stage1_raw/jw01257-o003_t005_nircam_f150w2-f162m/"},
    {"asn_name": "jw01257-o003_20260714t164831_image3_00009_asn.json", "dest_dir": "./processed_stage1_raw/jw01257-o003_t005_nircam_f150w2-f164n/"},
    {"asn_name": "jw01257-o003_20260714t164831_image3_00005_asn.json", "dest_dir": "./processed_stage1_raw/jw01257-o003_t005_nircam_f322w2-f323n/"},
    {"asn_name": "jw01257-o003_20260714t164831_image3_00008_asn.json", "dest_dir": "./processed_stage1_raw/jw01257-o003_t005_nircam_f444w-f466n/"},
    {"asn_name": "jw01257-o003_20260714t164831_image3_00007_asn.json", "dest_dir": "./processed_stage1_raw/jw01257-o003_t005_nircam_f444w-f470n/"}
]

if os.path.exists("./download_asn_members.py"):
    print("--- Starting Clean-Aligned Multi-Filter Download Matrix ---")

    for job in download_matrix:
        print(f"\nProcessing Group: {job['asn_name']}")
        print(f" -> Target Directory: {job['dest_dir']}")

        # We pass ONLY the filename as parameter 1. The utility script prepends its internal path cleanly.
        sys.argv = ["./download_asn_members.py", job["asn_name"], "1257", job["dest_dir"]]

        # Execute the download routine cleanly
        exec(open("./download_asn_members.py").read(), globals())
else:
    print("ERROR: download_asn_members.py utility script missing in root layout.")


## Step 4: Health-Check Logistics (Physical Footprint Verification Metrics)

In [ ]:
import os
import glob

target_filter_paths = [
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f150w2-f162m/mastDownload/JWST",
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f150w2-f164n/mastDownload/JWST",
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f322w2-f323n/mastDownload/JWST",
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f444w-f466n/mastDownload/JWST",
    "./processed_stage1_raw/jw01257-o003_t005_nircam_f444w-f470n/mastDownload/JWST"
]

total_recovered = 0
print("==========================================")
print("FILTER WORKSPACE RETRIEVAL VERIFICATION")
print("==========================================")

for path in target_filter_paths:
    if os.path.exists(path):
        files_in_filter = glob.glob(os.path.join(path, "jw*/*_uncal.fits"))
        folder_name = path.split('/')[2]
        print(f" -> Filter Workspace: {folder_name:<50} | Exposures Found: {len(files_in_filter)}")
        total_recovered += len(files_in_filter)

print("==========================================")
print(f"Total clean exposures matched for processing: {total_recovered} files")
print("==========================================")

## Step 5: Execute Complete Calibration Matrix (Detector1Pipeline & Image2Pipeline Workflow)

In [ ]:
# Execute Stage 1 and Stage 2 Pipelines sequentially over the data pools
pipeline_scripts = ["./run_stage_1_pipeline.py", "./run_stage_2_pipeline.py"]

for pipeline_run in pipeline_scripts:
    if os.path.exists(pipeline_run):
        print(f"\n--- Invoking Calibration Sequence: {pipeline_run} ---")
        exec(open(pipeline_run).read())
    else:
        print(f"Warning: Pipeline controller script {pipeline_run} not located.")

print("\nProcessing complete! All exposures successfully converted to standard *_cal.fits structural formats.")


--- Invoking Calibration Sequence: ./run_stage_1_pipeline.py ---
Located 32 raw files for structured pipeline processing.

--- Invoking Calibration Sequence: ./run_stage_2_pipeline.py ---
Located 32 raw files for pipeline processing inside ./processed_stage1_raw/

Processing Target: jw01257003001_02109_00002_nrcb1_uncal.fits
--- Running Stage 1: Detector Processing ---


2026-10-07 23:15:31,215 - CRDS - ERROR -  Error determining best reference for 'pars-darkcurrentstep'  =   No match found.
2026-10-07 23:15:31,220 - stpipe.step - INFO - PARS-JUMPSTEP parameters found: /home/kapranov/crds_cache/references/jwst/nircam/jwst_nircam_pars-jumpstep_0003.asdf
2026-10-07 23:15:31,240 - stpipe.step - INFO - PARS-DETECTOR1PIPELINE parameters found: /home/kapranov/crds_cache/references/jwst/nircam/jwst_nircam_pars-detector1pipeline_0008.asdf
2026-10-07 23:15:31,258 - stpipe.step - INFO - Detector1Pipeline instance created.
2026-10-07 23:15:31,259 - stpipe.step - INFO - GroupScaleStep instance created.
2026-10-07 23:15:31,260 - stpipe.step - INFO - DQInitStep instance created.
2026-10-07 23:15:31,261 - stpipe.step - INFO - EmiCorrStep instance created.
2026-10-07 23:15:31,261 - stpipe.step - INFO - SaturationStep instance created.
2026-10-07 23:15:31,262 - stpipe.step - INFO - IPCStep instance created.
2026-10-07 23:15:31,263 - stpipe.step - INFO - SuperBiasStep i


--- Running Stage 2: Image Processing for jw01257003001_02109_00002_nrcb1_rate.fits ---


2026-10-07 23:15:43,972 - stpipe.step - INFO - Step Image2Pipeline parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: None
  output_dir: ./processed_stage2_cal/
  output_ext: .fits
  output_use_model: False
  output_use_index: True
  save_results: True
  skip: False
  suffix: None
  search_output_file: True
  input_dir: ''
  save_bsub: False
  steps:
    bkg_subtract:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: False
      output_use_index: True
      save_results: False
      skip: True
      suffix: None
      search_output_file: True
      input_dir: ''
      bkg_list: None
      save_combined_background: False
      sigma: 3.0
      maxiters: None
      soss_source_percentile: 35.0
      soss_bkg_percentile: None
      wfss_mmag_extract: None
      wfss_mask: None
      wfss_maxiter: 5
      wfss_rms_stop: 0.0
      wfss_outlier_percent: 1.0
    assign_wcs:
      pre_hooks: []
 


Processing Target: jw01257003001_02109_00003_nrca2_uncal.fits
--- Running Stage 1: Detector Processing ---


...

2026-10-07 23:16:44,540 - stpipe.pipeline - INFO - Prefetching reference files for dataset: 'jw01257003001_02109_00003_nrca2_rate.fits' reftypes = ['area', 'camera', 'chromcorr', 'collimator', 'dflat', 'disperser', 'distortion', 'fflat', 'filteroffset', 'flat', 'fore', 'fpa', 'ifufore', 'ifupost', 'ifuslicer', 'msa', 'ote', 'photom', 'regions', 'sflat', 'specwcs', 'wavelengthrange']
2026-10-07 23:16:44,544 - CRDS - INFO -  Syncing 3 files
2026-10-07 23:16:44,545 - CRDS - INFO -  Fetching  /home/kapranov/crds_cache/references/jwst/nircam/jwst_nircam_area_0288.fits   16.8 M bytes  (1 / 3 files) (0 / 67.2 M bytes)
2026-10-07 23:16:54,938 - CRDS - INFO -  Fetching  /home/kapranov/crds_cache/references/jwst/nircam/jwst_nircam_distortion_0205.asdf   11.3 K bytes  (2 / 3 files) (16.8 M / 67.2 M bytes)
2026-10-07 23:16:55,326 - CRDS - INFO -  Fetching  /home/kapranov/crds_cache/references/jwst/nircam/jwst_nircam_flat_0665.fits   50.4 M bytes  (3 / 3 files) (16.8 M / 67.2 M bytes)
2026-10-07 23


Processing Target: jw01257003001_02109_00002_nrcb4_uncal.fits
--- Running Stage 1: Detector Processing ---


2026-10-07 23:17:29,373 - stpipe.step - INFO - Step Detector1Pipeline parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: None
  output_dir: ./processed_stage1_rate/
  output_ext: .fits
  output_use_model: False
  output_use_index: True
  save_results: True
  skip: False
  suffix: None
  search_output_file: True
  input_dir: ''
  save_calibrated_ramp: False
  steps:
    group_scale:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: False
      output_use_index: True
      save_results: False
      skip: False
      suffix: None
      search_output_file: True
      input_dir: ''
    dq_init:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: False
      output_use_index: True
      save_results: False
      skip: False
      suffix: None
      search_output_file: True
      input_dir: ''
      user_supplied


--- Running Stage 2: Image Processing for jw01257003001_02109_00002_nrcb4_rate.fits ---


2026-10-07 23:17:44,724 - stpipe.step - INFO - Step Image2Pipeline parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: None
  output_dir: ./processed_stage2_cal/
  output_ext: .fits
  output_use_model: False
  output_use_index: True
  save_results: True
  skip: False
  suffix: None
  search_output_file: True
  input_dir: ''
  save_bsub: False
  steps:
    bkg_subtract:
      pre_hooks: []
      post_hooks: []
      output_file: None
      output_dir: None
      output_ext: .fits
      output_use_model: False
      output_use_index: True
      save_results: False
      skip: True
      suffix: None
      search_output_file: True
      input_dir: ''
      bkg_list: None
      save_combined_background: False
      sigma: 3.0
      maxiters: None
      soss_source_percentile: 35.0
      soss_bkg_percentile: None
      wfss_mmag_extract: None
      wfss_mask: None
      wfss_maxiter: 5
      wfss_rms_stop: 0.0
      wfss_outlier_percent: 1.0
    assign_wcs:
      pre_hooks: []
 

<div style='padding: 15px; border: 2px solid #D9534F; border-radius: 5px; background-color: #FDF2F2; color: #B94A48;'>
    <h3 style='margin-top: 0; color: #D9534F; font-weight: bold;'>⚠️ STAGE 5: CRITICAL STORAGE & DISK SPACE WARNING</h3>
    <p style='font-size: 13px; line-height: 1.5;'>
        <strong>ATTENTION OPERATOR:</strong> JWST raw exposure calibrations require massive array allocations. To prevent local system drive saturation, <strong>DO NOT</strong> dump large batches of files into the ingestion path simultaneously.
    </p>
    <p style='font-size: 13px; line-height: 1.5;'>
        Process exposures sequentially by dropping <strong>ONLY ONE raw file at a time</strong> into the target directory path:<br>
        <code>./processed_stage1_raw/jw01257-o003_t005_nircam_f150w2-f164n/</code>
    </p>
    <p style='font-size: 13px; line-height: 1.5; margin-bottom: 0;'>
        Failure to partition inputs will expand the local cache directory footprint beyond system limits (<strong>~50 to 100 GB allocation spike</strong>), resulting in a hard kernel panic crash.
    </p>
</div>